In [1]:
!pip install scipy -q
from utils.distances_database_manager import get_distances, get_and_pivot_distances, get_data_with_max_distance
import dask
import dask.dataframe as dd
import pandas as pd
from IPython.display import display, Markdown
import networkx as nx
from tqdm import tqdm
import scipy as sp
from statistics import mean
from dask.diagnostics import ProgressBar
from dask.distributed import Client, progress, LocalCluster
from dask import delayed
import numpy as np

In [20]:
# path
database_path = '../data_parquet_starpep/'
dataset_path = '../datasets/StarPep/StarPep.csv'
dataset_path_amp = '../datasets/AMPDiscover/AMPDiscover(Training-Validation-Test).csv'

In [79]:
def load_csv(file_path):  
    data = dd.read_csv(file_path, usecols=['id', 'sequence'])
    return data

def load_parquet(file_path):
    data = dd.read_parquet(file_path, columns=['sequence'])
    return data['sequence'].unique().to_frame(name='sequence')

def find_missing_sequences(database_path, dataset_path):
    existing_sequences = load_parquet(database_path)
    new_sequences = load_csv(dataset_path)
    
    merged = dd.merge(new_sequences, existing_sequences, on='sequence', how='left', indicator=True)

    missing_sequences = merged[merged['_merge'] == 'left_only']
    result = missing_sequences.drop(columns=['_merge']).compute()
    
    return result

In [80]:
merged = find_missing_sequences(database_path, dataset_path_amp)

In [81]:
merged.shape

(24696, 2)

In [82]:
merged.head()

,id,sequence
9,seq20_train,GFIFHIIKG
39,seq74_train,LLPNLLKSL
44,seq85_train,GLRILLLKV
110,seq226_train,RWWRWRR
111,seq227_train,RRWWRRWRR


In [70]:
missing_sequences = merged[merged['_merge'] == 'left_only']
missing_sequences.shape

(24696, 2)

In [64]:
merged

,sequence,_merge
0,GLFDIIKNIFSGL,both
1,FLGALFKALSKLL,both
2,FFPVIGRILNGIL,both
3,TPFLLVGTQIDLR,both
4,GLLSRIKTLL,both
...,...,...
7811,MFTLKKSLLLLFFLGTISLSLCEEERDADEDEGEMTEEEVKRSVLG...,both
7812,FPLTCPTKWWKG,both
7813,VLSKSLCTPGCITGPLQTCYLCFPTFAKC,both
7814,GLLSGLKKVGKHVAKNVAVSLMDSLKCKISGDC,both


In [40]:
a = load_csv(dataset_path)
a

,sequence
npartitions=1,
,string
,...


In [39]:
b = load_parquet(database_path)
b

,sequence
npartitions=4,
,string
,...
,...
,...
,...


In [ ]:
merged = find_missing_sequences(database_path, dataset_path).tolist()

In [18]:
d = data.compute()

In [14]:
data

,id_x,sequence,id_y
npartitions=1,,,
,string,string,string
,...,...,...


In [22]:
d.shape

(45120, 2)